# SLAYER-OCR layout consensus v2
Run this notebook after obtaining one evidence ZIP from each pinned teacher: Qwen3-VL, DocLayout-YOLO and Surya Layout. Upload exactly those three ZIPs.
The notebook verifies archive safety, checksums, model revisions, run identity, page identity, frozen teacher configuration and consensus policy v2 before building COCO weak labels, a review queue and hard-example records. It never publishes the result.


In [ ]:
import json
import os
from pathlib import Path
import subprocess
import sys
from datetime import datetime, timezone

CODE_REVISION = '2c953866adc5ec7f8701072ab0a5eafdfd60a242'
EXPECTED_CONFIG = {'schema': 'slayer-layout-teacher-pilot-config-v1', 'status': 'frozen before teacher inference', 'frozen_at': '2026-09-28', 'scope': 'private weak-label pilot on historical Polish train pages; not benchmark evaluation', 'dataset': {'repo': 'PiotrSty/impact-psnc-polish-ocr', 'revision': 'c7cb156fb95d2880699c33725bbaf1fbc1008fea', 'metadata_path': 'pages/train/metadata.jsonl', 'metadata_sha256': '991ef94a2e2fb3ac519ec3582fa588b1a75a92a87b1b002468cf7c31d3c10427', 'available_pages': 65, 'selected_pages': 60, 'collections': 22, 'license': 'CC-BY-3.0', 'selection': 'collection-round-robin after SHA-256 rank', 'rank_salt': 'slayer-layout-teacher-pilot-v1'}, 'ontology': ['text_region', 'heading', 'table', 'figure', 'caption', 'marginalia', 'header', 'footer', 'page_number'], 'teachers': {'qwen3-vl-4b': {'model_repo': 'Qwen/Qwen3-VL-4B-Instruct', 'revision': 'ebb281ec70b05090aa6165b016eac8ec08e71b17', 'license': 'Apache-2.0', 'package': 'transformers==4.57.6', 'score_kind': 'neutral-unavailable', 'score': 0.5, 'max_new_tokens': 4096}, 'doclayout-yolo': {'model_repo': 'juliozhao/DocLayout-YOLO-DocStructBench', 'revision': '8c3299a30b8ff29a1503c4431b035b93220f7b11', 'license': 'Apache-2.0 weights; AGPL-3.0 inference package', 'package': 'doclayout-yolo==0.0.4', 'score_kind': 'model-confidence', 'threshold': 0.25, 'nms_iou': 0.45, 'image_size': 1024}, 'surya-layout2': {'model_repo': 'datalab-to/surya_layout2', 'revision': '0aee81d5fd9275c0582e545bf3a56944b1e75679', 'license': 'AI Pubs OpenRAIL-M', 'package': 'surya-ocr==0.22.1', 'score_kind': 'model-confidence when exposed; otherwise neutral-unavailable'}}, 'consensus': {'categories': ['text_region', 'heading', 'table', 'figure', 'caption', 'marginalia', 'header', 'footer', 'page_number'], 'quorum': 2, 'iou_threshold': 0.5, 'conflict_iou': 0.5, 'min_score': 0.0}, 'boundaries': {'test_or_validation_pages_used': 0, 'reference_text_sent_to_teachers': False, 'historical_spelling_modernized': False, 'teacher_outputs_are_ground_truth': False, 'images_or_references_in_evidence_zip': False, 'automatic_publication': False, 'release_requires_license_review': True}, 'claim_boundary': 'This pilot measures teacher agreement and review load. It does not establish label accuracy, RF-DETR improvement, or OCR SOTA.'}
EXPECTED_POLICY = {'categories': ['text_region', 'heading', 'table', 'figure', 'caption', 'marginalia', 'header', 'footer', 'page_number'], 'quorum': 2, 'iou_threshold': 0.5, 'conflict_iou': 0.5, 'min_score': 0.0, 'containment_threshold': 0.9, 'granularity_ratio': 2.0}

repo = Path("/content/OCR_engine")
if not repo.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/PiotrStyla/OCR_engine.git", str(repo)],
        check=True,
    )
subprocess.run(["git", "-C", str(repo), "fetch", "origin", CODE_REVISION], check=True)
subprocess.run(
    ["git", "-C", str(repo), "checkout", "--detach", CODE_REVISION], check=True
)
assert subprocess.check_output(
    ["git", "-C", str(repo), "rev-parse", "HEAD"], text=True
).strip() == CODE_REVISION
CONFIG = json.loads(
    (repo / "experiments/2026-09-28/slayer-layout-teacher-pilot-v1/config.json")
    .read_text(encoding="utf-8")
)
assert CONFIG == EXPECTED_CONFIG
POLICY = json.loads(
    (repo / "experiments/2026-09-28/slayer-layout-consensus-policy-v2.json")
    .read_text(encoding="utf-8")
)["consensus"]
assert POLICY == EXPECTED_POLICY
os.chdir(repo)
sys.path.insert(0, str(repo))
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
work = Path("/content") / ("slayer-layout-consensus-" + run_id)
print("PINNED_CONSENSUS_RUNTIME_OK", CODE_REVISION)


In [ ]:
from google.colab import files

uploaded = files.upload()
assert len(uploaded) == 3, "Upload exactly the three teacher evidence ZIPs."
upload_dir = work / "uploads"
upload_dir.mkdir(parents=True)
archives = []
for name, content in uploaded.items():
    assert Path(name).name == name and name.lower().endswith(".zip")
    target = upload_dir / name
    target.write_bytes(content)
    archives.append(target)
print("UPLOAD_OK", [path.name for path in archives])


In [ ]:
from training.slayer_layout_consensus_colab import combine_archives

result_archive, summary = combine_archives(
    archives, work / "combined", CONFIG, POLICY, code_revision=CODE_REVISION
)
print(json.dumps(summary, ensure_ascii=False, indent=2))
print("PRIVATE_CONSENSUS_READY", result_archive)


In [ ]:
from google.colab import files
files.download(str(result_archive))
